# Sparse Field Index

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Binary Search, Trees · **Difficulty/Frequency:** Common (6/10)

> **Dependency note.** The official answer uses `sortedcontainers.SortedDict`, a third-party package. This notebook builds the equivalent on the **standard library's `bisect`**, so it runs anywhere — and says plainly where the two differ.

## Concepts

**What this problem is really testing:**
- What a **database index** actually is, and specifically what makes one **sparse**
- **Binary search** for a lower bound, which is what turns a range query from O(n) into O(log n + k)
- Keeping **two structures consistent** — the index and the documents — through inserts *and* deletes

**First-principles primer — what is each piece?**

- **Index.** A second copy of your data, organised by the thing you query on. The documents are stored by `docId`; the index is stored by *field value*. Same information, different access path — exactly like the index at the back of a book versus the book itself.
- **Sparse index.** An index that **skips documents where the field is absent**. MongoDB's real `sparse: true` option does exactly this. `{"name": "Bob"}` has no `age`, so it simply never appears in an `age` index. Why bother? Because in a document database most documents lack most fields — indexing the absent ones would waste space on entries nobody can ever query.
- **Lower bound / `bisect_left`.** Given a sorted array and a value `x`, the position where `x` would be inserted to keep it sorted — found by binary search in O(log n). This is the single primitive that makes a range scan fast: jump straight to where the range starts, then walk forward.
- **Posting list / bucket.** Several documents can share a field value (two people aged 30). So the index maps `value -> a set of docIds`, not `value -> one docId`.

**Why O(log n + k) is the interesting requirement:**

That complexity is a *promise about two separate costs*:

| Part | Cost | What it is |
|---|---|---|
| `log n` | binary search | finding **where** the range begins |
| `k` | walking forward | reading out the **k** answers |

Crucially there is **no `n` term** — the work does not depend on how big the index is, only on how many results you asked for. A full scan (`for value in index: if low <= value <= high`) is O(n) and fails the requirement even though it returns the same answers.

**The detail that breaks most implementations:** `removeDocument` must **delete the bucket when it becomes empty**. Leave `{30: set()}` behind and the index grows forever, `rangeScan` walks over dead keys, and `30 in index` starts lying. The invariant to state aloud: *a value appears as a key if and only if at least one live document has that value.*

**Simple worked example.** `SparseIndex("age")` after indexing Alice (30), Bob (no age), Carol (25), Dave (30):

| sorted value | bucket |
|---|---|
| 25 | `{u3}` |
| 30 | `{u1, u4}` |

Bob is **absent entirely** — that is the sparseness. `rangeScan(25, 29)` binary-searches to position 0, reads `25`, sees `30 > 29`, and stops: **two steps**, not a scan of the whole index.

## Problem Statement

| Method | Behaviour |
|---|---|
| `SparseIndex(field)` | Index only on this field |
| `index_document(doc_id, doc)` | Add to the index — **no-op if the field is absent** |
| `remove_document(doc_id, doc)` | Remove from the index |
| `lookup(value)` | docIds whose field equals `value`, sorted |
| `range_scan(low, high, inclusive)` | docIds whose field lies in the range, ordered by value then docId |

**Example**

```python
idx = SparseIndex("age")
idx.index_document("u1", {"name": "Alice", "age": 30})
idx.index_document("u2", {"name": "Bob"})               # no-op: no "age"
idx.index_document("u3", {"name": "Carol", "age": 25})
idx.index_document("u4", {"name": "Dave", "age": 30})

idx.lookup(30)                             # -> ["u1", "u4"]
idx.range_scan(25, 29, inclusive=True)     # -> ["u3"]
idx.remove_document("u1", {"name": "Alice", "age": 30})
idx.lookup(30)                             # -> ["u4"]
```

**Constraints:** values are comparable; up to 10⁶ documents; `range_scan` must be **O(log n + k)**.

### Approach 1 — Naive (a flat list of `(value, docId)` pairs)

**Idea:** append every indexed pair to a list. Queries scan the whole thing.

Writing is trivially fast, and that is the trap — all the cost has been pushed onto every future read. With 10⁶ documents, a single `range_scan` touches a million entries to return maybe three.

**Time complexity:** O(1) `index_document`; **O(n)** `lookup`, `range_scan` and `remove_document`.

**Space complexity:** O(n).

In [ ]:
from typing import Any, Dict, List, Set, Tuple


class NaiveSparseIndex:
    """Baseline: correct, but every query is a full scan."""

    def __init__(self, field: str) -> None:
        self.field = field
        self.entries: List[Tuple[Any, str]] = []          # (value, doc_id)

    def index_document(self, doc_id: str, doc: dict) -> None:
        if self.field not in doc:
            return                                        # THE sparse rule
        self.entries.append((doc[self.field], doc_id))

    def remove_document(self, doc_id: str, doc: dict) -> None:
        if self.field not in doc:
            return                                        # must check here TOO
        pair = (doc[self.field], doc_id)
        if pair in self.entries:
            self.entries.remove(pair)                     # O(n) scan + O(n) shift

    def lookup(self, value: Any) -> List[str]:
        return sorted(d for v, d in self.entries if v == value)          # O(n)

    def range_scan(self, low: Any, high: Any, inclusive: bool = True) -> List[str]:
        if inclusive:
            hits = [(v, d) for v, d in self.entries if low <= v <= high]
        else:
            hits = [(v, d) for v, d in self.entries if low < v < high]
        return [d for v, d in sorted(hits)]                               # O(n log n)

### Approach 2 — Optimal (sorted key array + buckets, via `bisect`)

**Idea:** keep two things in step —

- `buckets: value -> set[doc_id]` — a plain dict, so `lookup` is O(1),
- `keys: [value, ...]` — the **same keys, kept sorted**, so a range query can binary-search its way in.

`range_scan` then becomes three moves: `bisect` to the start of the range, walk forward while still inside it, stop. That walk visits exactly the k matching values — never the whole index.

**Three details that carry the answer:**

- **The sparse check appears in *both* `index_document` and `remove_document`.** Omitting it from `remove` is the classic bug: a document with no field was never indexed, so trying to remove it must be a no-op, not a lookup of `doc[field]` that raises `KeyError`.
- **Empty buckets must be deleted.** The invariant is *a key exists iff a live document has that value*. Leaving `{30: set()}` behind leaks memory and makes `range_scan` walk over keys with nothing behind them.
- **The `inclusive` flag is handled at the boundaries, not in the loop.** `bisect_left` for an inclusive low bound, `bisect_right` for an exclusive one — the choice is made once, in O(log n), rather than tested per element.

**Honest complexity note.** `lookup` and `range_scan` hit the O(log n + k) target exactly. Insertion and deletion are **O(log n) to find the position but O(n) to shift the array** — `list.insert` is a memmove. That is the one place `sortedcontainers.SortedDict` (or a real balanced BST / B-tree, which is what a database actually uses) does better, achieving effectively O(log n) writes. The constant on a memmove is tiny, so the array wins in practice up to surprisingly large n — but the asymptotic difference is real and worth naming out loud.

**Time complexity:** **O(log n + k)** for `range_scan`; **O(1)** average for `lookup`; O(log n) search + O(n) shift for writes.

**Space complexity:** O(n) — one docId per indexed document, plus one key per distinct value.

In [ ]:
import bisect


class SparseIndex:
    """Sorted key array + value buckets. Only documents HAVING the field are indexed."""

    def __init__(self, field: str) -> None:
        self.field = field
        self.buckets: Dict[Any, Set[str]] = {}     # value -> set of doc ids  (O(1) lookup)
        self.keys: List[Any] = []                  # the same values, kept SORTED (binary search)

    # ---- writes ----------------------------------------------------------
    def index_document(self, doc_id: str, doc: dict) -> None:
        if self.field not in doc:
            return                                 # SPARSE: absent field -> not indexed at all
        value = doc[self.field]
        if value not in self.buckets:
            self.buckets[value] = set()
            bisect.insort(self.keys, value)        # O(log n) to place, O(n) to shift
        self.buckets[value].add(doc_id)            # a set => re-indexing is idempotent

    def remove_document(self, doc_id: str, doc: dict) -> None:
        if self.field not in doc:
            return                                 # never indexed => nothing to remove
        value = doc[self.field]
        bucket = self.buckets.get(value)
        if bucket is None:
            return
        bucket.discard(doc_id)                     # discard, not remove: absent id is not an error
        if not bucket:                             # INVARIANT: no empty buckets, ever
            del self.buckets[value]
            i = bisect.bisect_left(self.keys, value)
            self.keys.pop(i)

    # ---- reads -----------------------------------------------------------
    def lookup(self, value: Any) -> List[str]:
        return sorted(self.buckets.get(value, ()))          # O(1) hash hit + O(m log m) sort

    def range_scan(self, low: Any, high: Any, inclusive: bool = True) -> List[str]:
        # ONE binary search decides where to start - O(log n), independent of index size.
        i = bisect.bisect_left(self.keys, low) if inclusive else bisect.bisect_right(self.keys, low)
        out: List[str] = []
        while i < len(self.keys):                  # then walk forward over exactly the k matches
            v = self.keys[i]
            if (v > high) if inclusive else (v >= high):
                break                              # sorted => the first miss ends the scan
            out.extend(sorted(self.buckets[v]))    # values ascending, doc ids ascending within
            i += 1
        return out

    def __len__(self) -> int:
        return sum(len(b) for b in self.buckets.values())

### Follow-up 1 — a compound sparse index on two fields

**Idea:** index on `(field1_value, field2_value)` tuples, and index a document **only if both fields are present**.

Two things fall out of using a tuple key, and both are worth saying:

- **Python tuples compare lexicographically**, so `(30, "NY") < (30, "SF") < (31, "AK")` — sorting works with no custom comparator, and `bisect` keeps working unchanged.
- **The prefix rule.** A compound index on `(a, b)` can answer queries on `a` alone, or on `a` *and* `b` — but **not on `b` alone**, because entries with the same `b` are scattered across the whole ordering. This is exactly MongoDB's (and every SQL database's) compound-index prefix rule, and it is the reason index *order* matters when you design one.
- **Sparseness gets stricter.** Missing *either* field means the document is not indexed. So a compound sparse index covers fewer documents than either single-field index would.

**Time complexity:** identical to the single-field index.

**Space complexity:** O(n), with slightly larger keys.

In [ ]:
class CompoundSparseIndex:
    """Indexes on (field_a, field_b); a document must have BOTH fields to be indexed."""

    def __init__(self, field_a: str, field_b: str) -> None:
        self.fields = (field_a, field_b)
        self.buckets: Dict[Tuple[Any, Any], Set[str]] = {}
        self.keys: List[Tuple[Any, Any]] = []

    def _key(self, doc: dict):
        if any(f not in doc for f in self.fields):
            return None                            # BOTH must be present - stricter sparseness
        return tuple(doc[f] for f in self.fields)

    def index_document(self, doc_id: str, doc: dict) -> None:
        key = self._key(doc)
        if key is None:
            return
        if key not in self.buckets:
            self.buckets[key] = set()
            bisect.insort(self.keys, key)          # tuples sort lexicographically - free ordering
        self.buckets[key].add(doc_id)

    def remove_document(self, doc_id: str, doc: dict) -> None:
        key = self._key(doc)
        if key is None or key not in self.buckets:
            return
        self.buckets[key].discard(doc_id)
        if not self.buckets[key]:
            del self.buckets[key]
            self.keys.pop(bisect.bisect_left(self.keys, key))

    def lookup(self, value_a: Any, value_b: Any) -> List[str]:
        return sorted(self.buckets.get((value_a, value_b), ()))

    def prefix_scan(self, value_a: Any) -> List[str]:
        """Query the FIRST field alone - the prefix rule. Querying field_b alone is impossible."""
        i = bisect.bisect_left(self.keys, (value_a,))    # (a,) sorts before every (a, anything)
        out: List[str] = []
        while i < len(self.keys) and self.keys[i][0] == value_a:
            out.extend(sorted(self.buckets[self.keys[i]]))
            i += 1
        return out

### Follow-up 2 — surviving a stale `remove_document`

**Idea:** the API hands `remove_document` the *document*, and reads the field value out of it. That works only if the caller passes the document **exactly as it was indexed**. Pass an updated copy — `{"age": 31}` for a document indexed at 30 — and the removal silently targets the wrong bucket. The docId stays in bucket 30 forever, and `lookup(30)` keeps returning a document that no longer matches.

**The fix** is a reverse map `doc_id -> indexed value`, which makes the index the authority on what it stored rather than trusting the caller. It also enables `update_document`, which is the operation you actually want in a database, and it makes removal work from a docId alone.

The cost is one extra dict entry per document — the standard space-for-correctness trade.

**Time complexity:** unchanged; the reverse lookup is O(1).

**Space complexity:** O(n) extra.

In [ ]:
class RobustSparseIndex(SparseIndex):
    """Tracks what it actually indexed, so a stale or updated document can't corrupt it."""

    def __init__(self, field: str) -> None:
        super().__init__(field)
        self.indexed_value: Dict[str, Any] = {}    # doc_id -> the value we ACTUALLY stored

    def index_document(self, doc_id: str, doc: dict) -> None:
        if doc_id in self.indexed_value:           # re-indexing = update: drop the old entry first
            self._remove_by_id(doc_id)
        if self.field not in doc:
            return
        super().index_document(doc_id, doc)
        self.indexed_value[doc_id] = doc[self.field]

    def remove_document(self, doc_id: str, doc: dict = None) -> None:
        """The doc argument is now optional - we remember what we indexed."""
        self._remove_by_id(doc_id)

    def _remove_by_id(self, doc_id: str) -> None:
        if doc_id not in self.indexed_value:
            return
        value = self.indexed_value.pop(doc_id)     # OUR record, not the caller's claim
        bucket = self.buckets.get(value)
        if bucket is None:
            return
        bucket.discard(doc_id)
        if not bucket:
            del self.buckets[value]
            self.keys.pop(bisect.bisect_left(self.keys, value))

## Verification

Run the example from the statement, then the cases that separate a working index from a plausible one: sparseness on *both* paths, empty-bucket cleanup, exclusive bounds, string values, and the stale-document trap.

In [ ]:
import random

DOCS = {
    "u1": {"name": "Alice", "age": 30},
    "u2": {"name": "Bob"},                    # no age -> must never be indexed
    "u3": {"name": "Carol", "age": 25},
    "u4": {"name": "Dave", "age": 30},
}


def fresh(cls=SparseIndex):
    idx = cls("age")
    for did, doc in DOCS.items():
        idx.index_document(did, doc)
    return idx


# --- The example from the problem statement, on both implementations ---
for cls in (SparseIndex, NaiveSparseIndex, RobustSparseIndex):
    idx = fresh(cls)
    assert idx.lookup(30) == ["u1", "u4"], cls.__name__
    assert idx.range_scan(25, 29, inclusive=True) == ["u3"], cls.__name__
    idx.remove_document("u1", DOCS["u1"])
    assert idx.lookup(30) == ["u4"], cls.__name__

# --- Sparseness: the document with no field is invisible everywhere ---
idx = fresh()
assert "u2" not in idx.range_scan(-10**9, 10**9, inclusive=True)
assert len(idx) == 3, "only the 3 documents that HAVE an age are indexed"
idx.remove_document("u2", DOCS["u2"])         # removing an unindexed doc must be a silent no-op
assert len(idx) == 3

# --- Empty buckets must disappear, not linger ---
idx = fresh()
idx.remove_document("u3", DOCS["u3"])         # u3 was the only doc with age 25
assert 25 not in idx.buckets, "an emptied bucket must be deleted"
assert 25 not in idx.keys, "...from the sorted key array too"
assert idx.range_scan(20, 40, inclusive=True) == ["u1", "u4"]
idx.remove_document("u1", DOCS["u1"])
assert 30 in idx.buckets, "bucket 30 still holds u4 - do NOT delete it"
idx.remove_document("u4", DOCS["u4"])
assert idx.buckets == {} and idx.keys == [], "a fully drained index must be empty"
assert idx.range_scan(0, 100, inclusive=True) == []
assert idx.lookup(30) == []

# --- Range boundaries, inclusive and exclusive ---
idx = fresh()
assert idx.range_scan(25, 30, inclusive=True) == ["u3", "u1", "u4"]   # by value, then doc id
assert idx.range_scan(25, 30, inclusive=False) == []                  # 25 and 30 both excluded
assert idx.range_scan(24, 31, inclusive=False) == ["u3", "u1", "u4"]  # both now strictly inside
assert idx.range_scan(30, 30, inclusive=True) == ["u1", "u4"]         # a single-value range
assert idx.range_scan(31, 40, inclusive=True) == []                   # entirely above everything
assert idx.range_scan(0, 20, inclusive=True) == []                    # entirely below
assert idx.range_scan(40, 10, inclusive=True) == [], "an inverted range yields nothing"

# --- Re-indexing the same document is idempotent (a set absorbs it) ---
idx = fresh()
idx.index_document("u1", DOCS["u1"])
idx.index_document("u1", DOCS["u1"])
assert idx.lookup(30) == ["u1", "u4"], "no duplicate doc ids"

# --- String values sort and range-scan just as well as integers ---
s = SparseIndex("city")
for did, city in [("a", "Boston"), ("b", "Austin"), ("c", "Chicago"), ("d", "Austin")]:
    s.index_document(did, {"city": city})
assert s.lookup("Austin") == ["b", "d"]
assert s.range_scan("Austin", "Boston", inclusive=True) == ["b", "d", "a"]
assert s.range_scan("A", "B", inclusive=True) == ["b", "d"]

# --- The stale-document trap: plain removal is fooled, the robust index is not ---
plain = SparseIndex("age")
plain.index_document("x1", {"age": 30})
plain.remove_document("x1", {"age": 31})      # caller passes an UPDATED doc - wrong bucket
assert plain.lookup(30) == ["x1"], "the plain index is left with a stale entry (documented gap)"

robust = RobustSparseIndex("age")
robust.index_document("x1", {"age": 30})
robust.remove_document("x1", {"age": 31})     # the index trusts its own record instead
assert robust.lookup(30) == [], "the robust index removes the entry it actually made"

robust.index_document("x2", {"age": 30})
robust.index_document("x2", {"age": 40})      # re-index = update: the old entry must go
assert robust.lookup(30) == [] and robust.lookup(40) == ["x2"]
robust.index_document("x2", {"name": "no age now"})   # field removed -> drop from the index
assert robust.lookup(40) == [] and len(robust.buckets) == 0

# --- Compound index: both fields required, and the prefix rule ---
c = CompoundSparseIndex("age", "city")
c.index_document("p1", {"age": 30, "city": "SF"})
c.index_document("p2", {"age": 30, "city": "NY"})
c.index_document("p3", {"age": 25, "city": "SF"})
c.index_document("p4", {"age": 30})                     # missing city -> NOT indexed
c.index_document("p5", {"city": "SF"})                  # missing age  -> NOT indexed
assert c.lookup(30, "SF") == ["p1"]
assert c.prefix_scan(30) == ["p2", "p1"], "prefix query on the FIRST field; NY sorts before SF"
assert c.prefix_scan(25) == ["p3"]
assert c.prefix_scan(99) == []
assert sum(len(b) for b in c.buckets.values()) == 3, "p4 and p5 must be excluded"
c.remove_document("p1", {"age": 30, "city": "SF"})
assert c.lookup(30, "SF") == []

# --- Randomised: the fast index must agree with the naive one, through churn ---
random.seed(41)
for _ in range(60):
    docs = {}
    for i in range(60):
        d = {"name": f"n{i}"}
        if random.random() < 0.7:                        # 30% of documents have no age at all
            d["age"] = random.randint(0, 30)
        docs[f"d{i}"] = d

    fast, naive = SparseIndex("age"), NaiveSparseIndex("age")
    for did, doc in docs.items():
        fast.index_document(did, doc)
        naive.index_document(did, doc)

    for did in random.sample(list(docs), 20):            # delete a random fifth of them
        fast.remove_document(did, docs[did])
        naive.remove_document(did, docs[did])

    for v in range(0, 31, 5):
        assert fast.lookup(v) == naive.lookup(v), (v, "lookup")
    for lo, hi in [(0, 30), (5, 12), (10, 10), (25, 5), (-5, 40)]:
        for inc in (True, False):
            assert fast.range_scan(lo, hi, inc) == naive.range_scan(lo, hi, inc), (lo, hi, inc)

    # The invariant, checked directly: sorted keys, and never an empty bucket
    assert fast.keys == sorted(fast.buckets), "the key array must mirror the buckets, sorted"
    assert all(fast.buckets.values()), "no empty bucket may survive"

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Duplicate `index_document` for the same `(doc_id, value)`.** A set makes it idempotent, which is almost always the semantics you want: re-indexing an unchanged document should not double-count it. Say so explicitly rather than letting it be an accident — and note that a *list* bucket would silently produce duplicates in every query result.
- **A `remove_document` whose document no longer matches what was indexed.** Covered above by `RobustSparseIndex`. This is the deepest point in the question: the plain API **trusts the caller to hand back the original document**, and that trust is misplaced the moment the document is ever updated. A reverse `doc_id -> value` map makes the index self-sufficient, costs O(n) extra space, and is what a real storage engine does.
- **Concurrency.** Reads vastly outnumber writes on an index, so a **readers–writer lock** is the natural fit: many concurrent `lookup`/`range_scan`, exclusive access for `index_document`/`remove_document`. The subtlety is that `range_scan` returns a *list built while holding the lock* — do not hand back a live view of `keys`, or a concurrent write will invalidate it mid-iteration. For a read-mostly workload, copy-on-write of the key array lets readers run entirely lock-free, at the cost of an O(n) copy per write.
- **`null` values.** Three defensible policies: treat `null` as absent (do not index it — this is what MongoDB's sparse index does), index it as a value that sorts before everything, or reject it. All are fine; silently letting `None` into a mixed-type key array is not, because in Python 3 comparing `None` to an int raises `TypeError` and your `bisect` blows up. **Decide, document, and enforce at the boundary.**
- **Why a real database uses a B-tree, not this.** The array here does O(n) memmoves on write. A balanced BST fixes the asymptotics; a **B-tree** goes further by packing many keys per node so that one disk page holds hundreds of them — turning `log₂ n` random reads into `log₂₅₆ n`. On disk the number of *page fetches* is the only cost that matters, and that is the whole reason database indexes are B-trees rather than binary trees.

## Empirical complexity check

Compare the **full scan** (Approach 1) with the **binary-search + walk** (Approach 2), running a fixed number of narrow range scans against an index that doubles in size. The ranges are deliberately narrow, so k stays roughly constant and only the `log n` versus `n` term moves.

| Growth when the index doubles | What it means |
|---|---|
| ~2x | linear — every query touches every entry |
| ~1x | logarithmic — the binary search barely notices, and k did not change |

This is exactly the promise `O(log n + k)` makes: **query cost tracks the size of the answer, not the size of the index.**

Read the indexed row's **absolute times**, not just its ratios: they stay in the same few milliseconds while the scan climbs into the hundreds. `log n` only grows from 10 to 13 across this whole range, so its ratios sit near 1x with ordinary timing noise on top.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random

SCANS = 2000


def make_indexes(n):
    rng = random.Random(7)
    docs = [(f"d{i}", {"age": rng.randrange(n)}) for i in range(n)]
    fast, naive = SparseIndex("age"), NaiveSparseIndex("age")
    for did, doc in docs:
        fast.index_document(did, doc)
        naive.index_document(did, doc)
    lows = [rng.randrange(max(1, n - 10)) for _ in range(SCANS)]
    return (fast, naive, lows)


def run_scan_naive(fast, naive, lows):
    for lo in lows:
        naive.range_scan(lo, lo + 5, inclusive=True)      # O(n) every single query


def run_scan_indexed(fast, naive, lows):
    for lo in lows:
        fast.range_scan(lo, lo + 5, inclusive=True)       # O(log n + k)


benchmark(
    {"Approach 1 - full scan O(n)": run_scan_naive,
     "Approach 2 - bisect + walk O(log n + k)": run_scan_indexed},
    make_indexes,
    sizes=[1000, 2000, 4000, 8000],
    repeats=3,
)

## Patterns learned

- **An index is a second, differently-ordered copy of your data.** You store documents by id and index them by value, so that "which documents have value v?" stops being a scan. Every database index — and the [Inverted Index](../1.%20Inverted_Index/1.%20Inverted_Index.ipynb) next door — is this same move.
- **Sorted order is what makes ranges cheap.** Binary search jumps to where the answer starts; sortedness guarantees the first miss is the end. Without order, "between 25 and 29" has no choice but to look at everything.
- **`O(log n + k)` is a two-part promise.** *Find the start* in log time, *read out the answers* in linear-in-k time, and **never** touch anything else. Saying it as two terms is what shows you understand the shape of the algorithm.
- **A hash map and a sorted structure answer different questions.** Equality is a hash's job; ordering and ranges are a tree's or a sorted array's. This index carries both because the API asks both — and that is a normal, deliberate design, not redundancy.
- **State the invariant, then check every method against it.** *"A value is a key iff some live document has it."* That single sentence catches the empty-bucket leak, and it is exactly the argument an interviewer wants to hear.
- **Symmetric operations need symmetric guards.** The sparse check belongs in `remove` just as much as in `index`. Whenever you add a precondition on the way in, ask what the mirror operation must do about it.
- **Do not trust the caller to remember what you stored.** If correctness depends on being handed back the exact same input, keep your own record instead. One extra map buys immunity to a whole class of silent corruption.